In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Support Ticket Retrieval — Find Similar Tickets & Suggest Fixes

## 1. Project Overview

**Task:** Given a new support ticket, retrieve the most similar past tickets and suggest likely resolutions based on what worked before.

**Approach:** Build a ticket knowledge base → embed tickets with sentence-transformers → retrieve similar tickets via cosine similarity → use an LLM to synthesize a fix suggestion from past resolutions → compare semantic search vs keyword search.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for fix suggestion and evaluation
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store
- `LangChain` — text splitting and orchestration

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **semantic similarity** vs **keyword search** — when each wins |
| 2 | Build a **ticket embedding index** with metadata (product, severity, status) |
| 3 | **Retrieve** similar past tickets using cosine similarity |
| 4 | **Suggest fixes** by synthesizing resolutions from similar tickets |
| 5 | **Evaluate** retrieval quality with ground-truth ticket pairs |
| 6 | Handle edge cases: vague tickets, multi-issue tickets, no similar match |

## 3. Problem Statement

Support teams handle thousands of tickets, many of which describe the same underlying issue. When a new ticket arrives:

- An agent spends time diagnosing something that was solved last week
- Institutional knowledge lives in individual agents' heads
- Resolution quality varies depending on who picks up the ticket

A retrieval system can instantly surface: *"We've seen 4 tickets like this — here's what fixed them."*

## 4. Why This Matters

- **Response time** — median support resolution drops from hours to minutes
- **Consistency** — every agent sees the same institutional knowledge
- **Onboarding** — new agents get effective immediately
- **Deflection** — some tickets can be auto-resolved with a suggested fix

## 5. Semantic Similarity vs Keyword Search

This is the core concept of the notebook. Understanding when each approach works (and fails) is essential.

### Keyword Search (BM25 / TF-IDF)

Matches documents that share **exact words** with the query.

| Strengths | Weaknesses |
|-----------|------------|
| Fast and simple | Misses synonyms: "crash" ≠ "freeze" ≠ "hang" |
| Works for error codes, product names | Can't handle paraphrases |
| No model needed | Sensitive to word choice |
| Interpretable results | Short queries match too broadly |

### Semantic Search (Embeddings + Cosine Similarity)

Maps text to **dense vectors** where meaning-similar texts are close together.

| Strengths | Weaknesses |
|-----------|------------|
| "app crashes on login" ≈ "program freezes at startup" | Requires an embedding model |
| Handles paraphrases and synonyms automatically | Slower than keyword search |
| Works even when no words overlap | Black box — hard to explain rankings |
| Better for natural language queries | Can miss exact IDs/codes if not in training data |

### When to Use Which

| Scenario | Best Approach |
|----------|---------------|
| User types exact error code `ERR-4021` | Keyword |
| User describes symptoms in their own words | Semantic |
| Product name or specific feature | Keyword |
| "It's slow" / "things don't work" | Semantic |
| **Best practice** | **Hybrid: semantic + keyword** |

We'll implement both and compare side-by-side.

## 6. Pipeline Architecture

```
Past Tickets (title + description + resolution)
       │
  ┌────┴──────────────────────────────────────────┐
  │  Stage 1: TICKET KNOWLEDGE BASE                │
  │  • 30 realistic tickets across 5 products      │
  │  • Each: title, description, resolution, meta  │
  └────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────┐
  │  Stage 2: EMBED + INDEX                        │
  │  • Embed title+description with MiniLM         │
  │  • Store in ChromaDB with metadata             │
  └────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────┐
  │  Stage 3: RETRIEVAL                            │
  │  • Semantic: cosine similarity on embeddings   │
  │  • Keyword: naive TF-IDF baseline              │
  │  • Compare both on the same queries            │
  └────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────┐
  │  Stage 4: FIX SUGGESTION                       │
  │  • LLM reads similar tickets + resolutions     │
  │  • Synthesizes a recommended fix               │
  │  • Cites which past tickets informed the fix   │
  └────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────┐
  │  Stage 5: EVALUATION                           │
  │  • Recall@k and MRR on ground-truth pairs      │
  │  • Semantic vs keyword comparison              │
  └────────────────────────────────────────────────┘
```

## 7. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers scikit-learn

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers, scikit-learn")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 8. Imports

## 9. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5
TEMP_ANSWER = 0.1
TEMP_JUDGE = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Retrieval top-k: {TOP_K}")

## 10. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Ticket Knowledge Base

## 11. Build the Ticket Dataset

We create 30 realistic support tickets across 5 products. Each ticket has:
- **title** — short summary line
- **description** — user's actual words describing the problem
- **resolution** — how the issue was fixed
- **product** — which product it relates to
- **severity** — low / medium / high / critical
- **category** — bug, feature_request, how_to, account

The descriptions are written in varied natural language — some formal, some casual — to test semantic matching.

In [ ]:
TICKETS = [
    # --- CloudSync (file sync product) ---
    {"id": "TK-001", "product": "CloudSync", "severity": "critical", "category": "bug",
     "title": "Files not syncing after latest update",
     "description": "After updating CloudSync to v3.2, none of my files are syncing. The app shows a spinning icon but nothing uploads. I've tried restarting.",
     "resolution": "Known issue in v3.2. Clear the sync cache: Settings > Advanced > Clear Cache, then restart. If that fails, downgrade to v3.1.4 from the downloads page."},
    {"id": "TK-002", "product": "CloudSync", "severity": "high", "category": "bug",
     "title": "Sync stuck on 'Preparing'",
     "description": "CloudSync says 'Preparing to sync' for hours. Nothing ever uploads or downloads. My internet is fine — other apps work.",
     "resolution": "This happens when the sync index is corrupted. Delete the .cloudsync/index folder in your home directory and restart the app. It will rebuild the index."},
    {"id": "TK-003", "product": "CloudSync", "severity": "medium", "category": "bug",
     "title": "Duplicate files appearing after sync",
     "description": "I'm seeing duplicate copies of files with '(1)' appended. The duplicates appeared after I synced from two computers at the same time.",
     "resolution": "Conflict resolution created the duplicates. Delete the duplicates and enable Settings > Sync > 'Lock file during edit' to prevent future conflicts."},
    {"id": "TK-004", "product": "CloudSync", "severity": "low", "category": "how_to",
     "title": "How to exclude folders from sync",
     "description": "I want to stop syncing my node_modules and .git folders. They take up too much bandwidth.",
     "resolution": "Go to Settings > Selective Sync > Add Exclusion Pattern. Add 'node_modules' and '.git'. Changes apply on next sync cycle."},
    {"id": "TK-005", "product": "CloudSync", "severity": "high", "category": "bug",
     "title": "CloudSync crashes on startup with error ERR-4021",
     "description": "Every time I open CloudSync, it crashes immediately. The error log shows ERR-4021: database lock timeout. Running macOS Ventura.",
     "resolution": "ERR-4021 means the local database is locked by another process. Run 'lsof | grep cloudsync' to find the zombie process and kill it. Then restart CloudSync."},
    {"id": "TK-006", "product": "CloudSync", "severity": "medium", "category": "how_to",
     "title": "Set up CloudSync on a new computer",
     "description": "I got a new laptop. How do I set up CloudSync and get all my files without re-downloading everything?",
     "resolution": "Install CloudSync from cloudvault.io/download. Sign in with your account. Choose 'Smart Sync' to download files on demand instead of all at once. Only files you open will download."},

    # --- BillingPortal (payment / subscription) ---
    {"id": "TK-007", "product": "BillingPortal", "severity": "critical", "category": "account",
     "title": "Charged twice for the same subscription",
     "description": "I see two charges of $29.99 on my credit card for this month. I only have one account. Please refund the duplicate.",
     "resolution": "Duplicate charge confirmed in the billing system. Refund issued for $29.99 — allow 3-5 business days. Root cause: retry logic triggered a second charge during a gateway timeout."},
    {"id": "TK-008", "product": "BillingPortal", "severity": "high", "category": "account",
     "title": "Cannot update credit card",
     "description": "When I try to update my payment method, the page shows a blank white screen. I'm on Chrome 120.",
     "resolution": "Known issue with Chrome 120 and our payment iframe. Workaround: use Firefox or Safari. Fix shipping in BillingPortal v2.8 next week."},
    {"id": "TK-009", "product": "BillingPortal", "severity": "medium", "category": "how_to",
     "title": "How to cancel my subscription",
     "description": "I want to cancel my plan. I don't see a cancel button anywhere in the settings.",
     "resolution": "Go to Account > Subscription > click 'Manage Plan'. Scroll down to 'Cancel Subscription'. Note: you keep access until the end of the billing period."},
    {"id": "TK-010", "product": "BillingPortal", "severity": "high", "category": "account",
     "title": "Invoice shows wrong company name",
     "description": "My invoices still show my old company name. I updated it in settings last month but the invoices didn't change.",
     "resolution": "Billing details and account details are separate. Go to Billing > Invoice Settings > Company Name to update invoices specifically. Past invoices cannot be changed — contact support for a credit note."},
    {"id": "TK-011", "product": "BillingPortal", "severity": "low", "category": "feature_request",
     "title": "Request: pay annually for discount",
     "description": "Is there a way to pay yearly instead of monthly? I'd expect a discount for annual payment.",
     "resolution": "Annual billing is available on Standard and Enterprise plans. Go to Billing > Switch to Annual. Annual plans get a 20% discount. Prorated credit applied for remaining monthly period."},
    {"id": "TK-012", "product": "BillingPortal", "severity": "critical", "category": "bug",
     "title": "Account locked after payment failure",
     "description": "My account got locked because my credit card expired. I've updated the card but I still can't log in. Says 'account suspended'.",
     "resolution": "After updating payment, account reactivation takes up to 1 hour. If still locked after 1 hour: contact support with your account email to manually unlock. We're fixing the auto-reactivation delay."},

    # --- DevConsole (developer tools / API) ---
    {"id": "TK-013", "product": "DevConsole", "severity": "high", "category": "bug",
     "title": "API returns 500 error on POST /users",
     "description": "Getting HTTP 500 Internal Server Error when creating users via POST /v2/users. Started happening today. GET requests work fine.",
     "resolution": "The user creation service had a deployment issue. The API team rolled back at 14:30 UTC. POST /users is working again. If you still see errors, the request may be hitting a stale cache — add a unique query param."},
    {"id": "TK-014", "product": "DevConsole", "severity": "medium", "category": "bug",
     "title": "Rate limited even though I'm under the limit",
     "description": "I'm getting 429 Too Many Requests but I'm only making about 50 requests per minute. My plan allows 300/min.",
     "resolution": "Rate limits are per-API-key, not per-account. Check if you have multiple services using the same key. Also, burst rate is 10 requests/second even on the 300/min plan. Spread requests with 100ms delays."},
    {"id": "TK-015", "product": "DevConsole", "severity": "medium", "category": "how_to",
     "title": "How to authenticate with OAuth2",
     "description": "The docs mention OAuth2 but I can't figure out the flow. I need to let users sign in through our app.",
     "resolution": "Use the Authorization Code flow: 1) Redirect user to /oauth/authorize with client_id and redirect_uri. 2) User approves. 3) Exchange the code at /oauth/token for an access token. See docs: developers.cloudvault.io/oauth"},
    {"id": "TK-016", "product": "DevConsole", "severity": "low", "category": "how_to",
     "title": "Webhook payload format documentation",
     "description": "Where can I find the full webhook payload schema? The examples in the docs only show partial data.",
     "resolution": "Full webhook schemas are in the API reference: developers.cloudvault.io/webhooks/schemas. Each event type has a separate schema page. You can also enable 'debug mode' in DevConsole > Webhooks to log full payloads."},
    {"id": "TK-017", "product": "DevConsole", "severity": "high", "category": "bug",
     "title": "SDK throws 'InvalidSignature' on webhook verify",
     "description": "Using the Python SDK to verify webhook signatures but getting InvalidSignature errors on every request. The secret is correct — I copied it from the dashboard.",
     "resolution": "Common cause: the SDK expects the raw request body (bytes), not the parsed JSON. Pass request.body (not request.json()) to verify_signature(). Also check for trailing whitespace in the webhook secret."},
    {"id": "TK-018", "product": "DevConsole", "severity": "critical", "category": "bug",
     "title": "API responses extremely slow (>10s)",
     "description": "All API calls are taking 10-15 seconds to respond. This started about 2 hours ago. Our app is basically unusable.",
     "resolution": "Infrastructure incident affecting US-East region. Requests were being routed through a degraded data center. The team has rerouted traffic. If still slow, switch to the EU endpoint temporarily: api-eu.cloudvault.io."},

    # --- TeamChat (team messaging) ---
    {"id": "TK-019", "product": "TeamChat", "severity": "high", "category": "bug",
     "title": "Notifications not working on mobile",
     "description": "I'm not getting any push notifications from TeamChat on my iPhone. Desktop notifications work fine.",
     "resolution": "Check iOS Settings > TeamChat > Notifications > ensure 'Allow Notifications' is ON. Also in TeamChat mobile: Settings > Notifications > ensure 'Push' is enabled. If still broken, delete and reinstall the app — this refreshes the push token."},
    {"id": "TK-020", "product": "TeamChat", "severity": "medium", "category": "bug",
     "title": "Cannot upload images in chat",
     "description": "When I try to drag and drop an image into a chat, nothing happens. The attachment button also doesn't work. Other file types upload fine.",
     "resolution": "Image uploads were temporarily disabled due to a storage issue. This is now resolved. Clear your browser cache and reload. If using the desktop app, restart it."},
    {"id": "TK-021", "product": "TeamChat", "severity": "low", "category": "feature_request",
     "title": "Request: message threading like Slack",
     "description": "We really need threaded replies. Conversations get messy when multiple topics are discussed in the same channel.",
     "resolution": "Threading is on our Q3 roadmap. In the meantime, you can use 'Topics' (right-click a message > Start Topic) to branch a conversation without cluttering the main channel."},
    {"id": "TK-022", "product": "TeamChat", "severity": "medium", "category": "how_to",
     "title": "How to set up a guest account for external contractor",
     "description": "We need to give our freelance designer access to one channel without seeing the rest of our workspace.",
     "resolution": "Go to Admin > People > Invite Guest. Enter their email and select which channels they can see. Guest accounts have no access to other channels, files, or member lists outside their assigned channels."},
    {"id": "TK-023", "product": "TeamChat", "severity": "high", "category": "bug",
     "title": "Messages showing as 'sent' but others don't receive them",
     "description": "I'm sending messages that show a checkmark on my end, but my teammates say they never received them. This has been happening for two days.",
     "resolution": "This indicates a WebSocket connection issue. Your messages are queued locally but not being delivered. Fix: log out and back in (this forces a new WS connection). If persists, check if your corporate firewall blocks WebSocket (port 443/wss)."},

    # --- AnalyticsDash (analytics dashboard) ---
    {"id": "TK-024", "product": "AnalyticsDash", "severity": "high", "category": "bug",
     "title": "Dashboard shows wrong data after timezone change",
     "description": "I changed my timezone to UTC+2 and now all my charts show data shifted by 2 hours. Daily totals are also wrong.",
     "resolution": "Known issue: timezone changes don't recalculate historical aggregations. Workaround: go to Settings > Data > click 'Recalculate Aggregations'. This takes 5-10 minutes. We're fixing this to happen automatically."},
    {"id": "TK-025", "product": "AnalyticsDash", "severity": "medium", "category": "bug",
     "title": "CSV export only downloads first 1000 rows",
     "description": "When I export my data to CSV, I only get 1000 rows. My dataset has 50,000 rows. The export should include everything.",
     "resolution": "The default CSV export is limited to 1000 rows for performance. For full exports: use the API endpoint GET /export?format=csv&limit=0 (limit=0 means no limit). Or go to Settings > Export > set 'Default Row Limit' to your desired amount."},
    {"id": "TK-026", "product": "AnalyticsDash", "severity": "medium", "category": "how_to",
     "title": "How to create a shared dashboard",
     "description": "I want to create a dashboard that my whole team can see. Right now dashboards seem to be private.",
     "resolution": "Open the dashboard > click the Share icon (top right) > choose 'Team' or specific members. You can set permissions: 'View only' or 'Can edit'. Shared dashboards appear in the team's 'Shared' tab."},
    {"id": "TK-027", "product": "AnalyticsDash", "severity": "low", "category": "how_to",
     "title": "How to embed a chart on external website",
     "description": "Can I put one of my AnalyticsDash charts on our company website? Like an iframe or something.",
     "resolution": "Yes. Open the chart > click ⋯ menu > 'Embed'. Copy the iframe code. Works on Standard and Enterprise plans. Charts update in real-time. You can also use the public link for sharing without embedding."},
    {"id": "TK-028", "product": "AnalyticsDash", "severity": "critical", "category": "bug",
     "title": "Dashboard loading infinitely — blank page",
     "description": "My main dashboard just shows a white screen with a loading spinner. It's been like this for hours. Other dashboards load fine.",
     "resolution": "This usually means one widget is crashing the render. Open the dashboard in safe mode by adding ?safe=true to the URL. This loads widgets one at a time. Find and remove the broken widget, then reload normally."},
    {"id": "TK-029", "product": "AnalyticsDash", "severity": "high", "category": "bug",
     "title": "Data not updating — shows yesterday's numbers",
     "description": "My real-time dashboard hasn't updated since yesterday. The last data point is from 11pm last night.",
     "resolution": "Check the data source connection: Settings > Data Sources. If the status shows 'Disconnected', click 'Reconnect'. Common cause: the API key for the data source expired. Generate a new key and update it in Data Sources."},
    {"id": "TK-030", "product": "AnalyticsDash", "severity": "medium", "category": "bug",
     "title": "Filters reset every time I refresh the page",
     "description": "I set date range and product filters on my dashboard, but when I refresh the page, they all go back to defaults.",
     "resolution": "Filters are not saved by default. To persist them: set your filters, then click the bookmark icon next to the filter bar — this saves the current filter state as a 'View'. You can create multiple saved views."},
]

print(f"Total tickets: {len(TICKETS)}")
print(f"\nProducts: {Counter(t['product'] for t in TICKETS)}")
print(f"Severities: {Counter(t['severity'] for t in TICKETS)}")
print(f"Categories: {Counter(t['category'] for t in TICKETS)}")

print(f"\nSample ticket:")
t = TICKETS[0]
print(f"  ID: {t['id']}")
print(f"  Title: {t['title']}")
print(f"  Description: {t['description'][:80]}...")
print(f"  Resolution: {t['resolution'][:80]}...")

## 12. Inspect the Dataset

In [ ]:
print("TICKET KNOWLEDGE BASE")
print("=" * 90)
for t in TICKETS:
    sev_icon = {"critical": "!!!", "high": "!! ", "medium": "!  ", "low": ".  "}.get(t["severity"], "?")
    print(f"  {t['id']} {sev_icon} [{t['product']:15s}] [{t['category']:>15}] {t['title'][:50]}")

---

# Part B — Embedding & Indexing

## 13. Prepare Ticket Text for Embedding

We embed `title + description` (what the user wrote), not the resolution. The resolution is what we *retrieve after* finding a match.

In [ ]:
def ticket_to_text(ticket: dict) -> str:
    """Combine title and description for embedding."""
    return f"{ticket['title']}\n{ticket['description']}"


ticket_texts = [ticket_to_text(t) for t in TICKETS]
ticket_metadatas = [
    {
        "ticket_id": t["id"],
        "product": t["product"],
        "severity": t["severity"],
        "category": t["category"],
        "title": t["title"],
        "resolution": t["resolution"],
    }
    for t in TICKETS
]

print(f"Prepared {len(ticket_texts)} ticket texts for embedding")
print(f"Avg text length: {sum(len(t) for t in ticket_texts) // len(ticket_texts)} chars")
print(f"Shortest: {min(len(t) for t in ticket_texts)} chars")
print(f"Longest: {max(len(t) for t in ticket_texts)} chars")

## 14. Build the ChromaDB Index

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("support_tickets")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="support_tickets",
    metadata={"hnsw:space": "cosine"},
)

vectors = embeddings.embed_documents(ticket_texts)
collection.add(
    documents=ticket_texts,
    embeddings=vectors,
    metadatas=ticket_metadatas,
    ids=[t["id"] for t in TICKETS],
)

print(f"Indexed {collection.count()} tickets in ChromaDB")
print(f"Embedding dim: {len(vectors[0])}")

---

# Part C — Retrieval: Semantic vs Keyword

## 15. Semantic Retrieval

This is the core similarity search — embed the query and find the nearest tickets by cosine similarity.

In [ ]:
def retrieve_semantic(query: str, top_k: int = TOP_K,
                      where: Optional[dict] = None) -> list[dict]:
    """Retrieve similar tickets using embedding cosine similarity."""
    query_vector = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_vector], "n_results": top_k}
    if where:
        kwargs["where"] = where
    results = collection.query(**kwargs)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
            "similarity": 1 - results["distances"][0][i],
        })
    return hits


def display_hits(hits: list[dict], method: str = "Semantic"):
    """Pretty-print retrieval results."""
    print(f"\n  [{method}] Top {len(hits)} results:")
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        sim = h.get("similarity", h.get("score", 0))
        print(f"    {i}. sim={sim:.3f} | {m['ticket_id']} [{m['product']}] {m['title'][:50]}")


# Test semantic retrieval
q = "My files won't upload and the app just spins"
print(f"Query: {q}")
hits = retrieve_semantic(q, top_k=5)
display_hits(hits, "Semantic")

## 16. Keyword Retrieval (TF-IDF Baseline)

For comparison, we implement a simple TF-IDF keyword search. This shows exactly where keyword matching fails and semantic search succeeds.

In [ ]:
# Build TF-IDF index
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
)
tfidf_matrix = tfidf_vectorizer.fit_transform(ticket_texts)
print(f"TF-IDF matrix: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")


def retrieve_keyword(query: str, top_k: int = TOP_K) -> list[dict]:
    """Retrieve similar tickets using TF-IDF keyword matching."""
    query_vec = tfidf_vectorizer.transform([query])
    scores = sklearn_cosine(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    hits = []
    for idx in top_indices:
        t = TICKETS[idx]
        hits.append({
            "text": ticket_texts[idx],
            "metadata": ticket_metadatas[idx],
            "similarity": float(scores[idx]),
        })
    return hits


# Test keyword retrieval
q = "My files won't upload and the app just spins"
print(f"\nQuery: {q}")
hits_kw = retrieve_keyword(q, top_k=5)
display_hits(hits_kw, "Keyword")

## 17. Side-by-Side Comparison

Now we run the same queries through both methods and see where they agree and disagree.

In [ ]:
comparison_queries = [
    # Paraphrase: should match TK-001 (files not syncing)
    # Semantic wins — no keyword overlap with 'syncing'
    "The app is stuck and nothing is being uploaded to the cloud",

    # Exact error code: keyword should win
    "ERR-4021 crash on startup",

    # Synonym: 'frozen' vs 'stuck' vs 'spin' — semantic wins
    "Program completely frozen, can't do anything",

    # How-to phrased casually — semantic handles intent
    "I need to stop syncing some folders, too much data",

    # Billing issue described differently
    "You guys billed me twice this month, what the heck",

    # Vague complaint — semantic may capture intent
    "Everything is super slow today",
]

print("SEMANTIC vs KEYWORD — SIDE-BY-SIDE")
print("=" * 90)

for query in comparison_queries:
    sem = retrieve_semantic(query, top_k=3)
    kw = retrieve_keyword(query, top_k=3)

    print(f"\n  Q: {query}")
    print(f"  {'Semantic':^43s} | {'Keyword':^43s}")
    print(f"  {'-'*43} | {'-'*43}")

    for i in range(3):
        s_id = sem[i]["metadata"]["ticket_id"]
        s_sim = sem[i]["similarity"]
        s_title = sem[i]["metadata"]["title"][:25]

        k_id = kw[i]["metadata"]["ticket_id"]
        k_sim = kw[i]["similarity"]
        k_title = kw[i]["metadata"]["title"][:25]

        match = "=" if s_id == k_id else " "
        print(f"  {i+1}. {s_id} {s_sim:.3f} {s_title:25s} |{match}{i+1}. {k_id} {k_sim:.3f} {k_title:25s}")

## 18. Filtered Retrieval by Product

In a real system, you often know which product the user is asking about. Filtering narrows the search space.

In [ ]:
q = "My dashboard isn't showing the latest data"
print(f"Q: {q}")

print("\nUnfiltered:")
hits = retrieve_semantic(q)
display_hits(hits[:3], "All products")

print("\nFiltered to AnalyticsDash:")
hits_filtered = retrieve_semantic(q, where={"product": "AnalyticsDash"})
display_hits(hits_filtered[:3], "AnalyticsDash only")

---

# Part D — Fix Suggestion Pipeline

## 19. Suggest Fixes from Similar Tickets

Now the core QA step: given similar past tickets and their resolutions, synthesize a recommended fix for the new ticket.

In [ ]:
FIX_SYSTEM = """You are a support agent assistant. Given a new support ticket and similar past tickets with their resolutions, suggest the most likely fix.

Rules:
1. Synthesize a clear, actionable fix from the past resolutions.
2. Cite which past ticket(s) inform your suggestion as [Ticket: TK-XXX].
3. If the past tickets suggest different fixes, list them in order of likelihood.
4. If none of the past tickets are relevant enough, say so clearly.
5. Be concise — support agents need to act fast."""


def suggest_fix(new_ticket: str,
                where: Optional[dict] = None) -> dict:
    """Full pipeline: retrieve similar tickets -> suggest fix."""
    hits = retrieve_semantic(new_ticket, top_k=TOP_K, where=where)

    # Format past tickets for the LLM
    past = []
    for h in hits:
        m = h["metadata"]
        past.append(
            f"[Ticket: {m['ticket_id']}] (similarity: {h['similarity']:.2f})\n"
            f"Title: {m['title']}\n"
            f"Resolution: {m['resolution']}"
        )
    past_text = "\n\n".join(past)

    prompt = (
        f"NEW TICKET:\n{new_ticket}\n\n"
        f"SIMILAR PAST TICKETS:\n{past_text}\n\n"
        "Return ONLY JSON:\n"
        '{"suggested_fix": "step-by-step fix with [Ticket: TK-XXX] citations",'
        ' "confidence": "high|medium|low",'
        ' "confidence_reason": "why",'
        ' "tickets_used": ["TK-XXX"],'
        ' "alternative_fixes": ["if applicable"]}'
    )

    resp = ask(prompt, system=FIX_SYSTEM, temperature=TEMP_ANSWER)
    result = parse_json(resp)
    if not result:
        result = {
            "suggested_fix": resp,
            "confidence": "unknown",
            "confidence_reason": "LLM did not return valid JSON",
            "tickets_used": [h["metadata"]["ticket_id"] for h in hits[:3]],
            "alternative_fixes": [],
        }
    result["similar_tickets"] = hits
    return result


def display_fix(ticket_text: str, result: dict):
    conf = result.get("confidence", "?")
    badge = {"high": "HIGH", "medium": "MED", "low": "LOW"}.get(conf, "???")

    print(f"\n{'='*80}")
    print(f"NEW TICKET: {ticket_text[:80]}...")
    print(f"Confidence: {badge}")
    print(f"{'='*80}")
    print(f"\nSUGGESTED FIX:")
    wrap_print(result.get("suggested_fix", ""))
    print(f"\n  Based on: {', '.join(result.get('tickets_used', []))}")
    if result.get("alternative_fixes"):
        print(f"\n  Alternatives:")
        for alt in result["alternative_fixes"]:
            print(f"    - {textwrap.shorten(str(alt), width=80, placeholder='...')}")


print("Fix suggestion pipeline ready")

## 20. Test Fix Suggestions

In [ ]:
t1 = """Subject: App won't sync anymore
My CloudSync app stopped uploading files yesterday. It keeps showing a loading spinner but nothing happens. I've restarted my computer twice."""

r1 = suggest_fix(t1)
display_fix(t1, r1)

In [ ]:
t2 = """Subject: Double-charged on my account
I was charged $29.99 twice on March 5th. I only have one subscription. Please fix this and refund the extra charge."""

r2 = suggest_fix(t2)
display_fix(t2, r2)

In [ ]:
t3 = """Subject: API is extremely slow
All our API calls to CloudVault have been taking over 10 seconds since this morning. Our production app depends on this. Please escalate immediately."""

r3 = suggest_fix(t3)
display_fix(t3, r3)

In [ ]:
t4 = """Subject: Webhook verification failing
I'm using the Python SDK to verify webhook signatures but it keeps returning False even though I've triple-checked the secret key."""

r4 = suggest_fix(t4)
display_fix(t4, r4)

## 21. Edge Case: Vague Ticket

In [ ]:
t_vague = """Subject: Help needed
Something isn't working right. Can someone look into this?"""

r_vague = suggest_fix(t_vague)
display_fix(t_vague, r_vague)
print(f"\n  -> Expected LOW confidence. The ticket is too vague for a specific fix.")

## 22. Edge Case: No Similar Ticket

In [ ]:
t_novel = """Subject: GDPR data export request
Under GDPR Article 15, I request a full export of all personal data you hold about me.
My account email is user@example.com. Please provide this within 30 days as required by law."""

r_novel = suggest_fix(t_novel)
display_fix(t_novel, r_novel)
print(f"\n  -> Expected LOW confidence. No GDPR tickets exist in our knowledge base.")

## 23. Batch Processing

In [ ]:
batch_tickets = [
    "I can't receive messages from my team. They send but I don't see anything.",
    "How do I set up OAuth for my app?",
    "My CSV export is missing most of the data, only 1000 rows",
    "Dashboard is blank, infinite loading screen",
    "Want to invite an external person to just one channel",
]

print("BATCH FIX SUGGESTIONS")
print("=" * 90)

for ticket in batch_tickets:
    result = suggest_fix(ticket)
    conf = result.get("confidence", "?")
    badge = {"high": "HIGH", "medium": "MED ", "low": "LOW "}.get(conf, "??? ")
    fix = result.get("suggested_fix", "")
    tickets = ", ".join(result.get("tickets_used", [])[:3])

    print(f"\n  [{badge}] Q: {ticket[:60]}")
    print(f"       Fix: {textwrap.shorten(fix, width=85, placeholder='...')}")
    print(f"       Tickets: {tickets}")

---

# Part E — Retrieval Evaluation

## 24. Ground-Truth Evaluation Set

We define which tickets a new query *should* match, then measure Recall@k and MRR for both semantic and keyword retrieval.

In [ ]:
EVAL_SET = [
    # Paraphrased versions of existing tickets
    {"query": "CloudSync won't upload anything, just keeps loading",
     "expected_ids": ["TK-001", "TK-002"]},
    {"query": "Getting duplicate copies of my synced files",
     "expected_ids": ["TK-003"]},
    {"query": "CloudSync crashes at launch with a database error",
     "expected_ids": ["TK-005"]},
    {"query": "You guys charged my credit card two times",
     "expected_ids": ["TK-007"]},
    {"query": "I can't change my payment information, page is broken",
     "expected_ids": ["TK-008"]},
    {"query": "Where's the cancel button for my subscription?",
     "expected_ids": ["TK-009"]},
    {"query": "My account is locked even though I fixed the payment",
     "expected_ids": ["TK-012"]},
    {"query": "Creating users through the API gives a server error",
     "expected_ids": ["TK-013"]},
    {"query": "Getting rate limited way below my plan's limit",
     "expected_ids": ["TK-014"]},
    {"query": "Phone notifications aren't working for TeamChat",
     "expected_ids": ["TK-019"]},
    {"query": "My teammates can't see the messages I send",
     "expected_ids": ["TK-023"]},
    {"query": "Charts show wrong times after changing timezone",
     "expected_ids": ["TK-024"]},
    {"query": "CSV export is limited to 1000 rows, need full data",
     "expected_ids": ["TK-025"]},
    {"query": "Dashboard won't load, just a white page with spinner",
     "expected_ids": ["TK-028"]},
]

print(f"Evaluation set: {len(EVAL_SET)} query-ticket pairs")

## 25. Run Evaluation: Semantic vs Keyword

In [ ]:
def evaluate_retrieval(eval_set: list[dict], retriever, method_name: str,
                       top_k: int = 5) -> dict:
    """Compute Recall@k and MRR for a retriever."""
    hits_at_k = 0
    reciprocal_ranks = []
    details = []

    for item in eval_set:
        hits = retriever(item["query"], top_k=top_k)
        retrieved_ids = [h["metadata"]["ticket_id"] for h in hits]
        expected = item["expected_ids"]

        # Check if any expected ticket is in the retrieved set
        found_rank = None
        for rank, rid in enumerate(retrieved_ids, 1):
            if rid in expected:
                found_rank = rank
                break

        if found_rank is not None:
            hits_at_k += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)

        details.append({
            "query": item["query"],
            "expected": expected,
            "top_3": retrieved_ids[:3],
            "hit": found_rank is not None,
            "rank": found_rank,
        })

    n = len(eval_set)
    return {
        "method": method_name,
        "recall_at_k": hits_at_k / n,
        "mrr": sum(reciprocal_ranks) / n,
        "n": n,
        "k": top_k,
        "details": details,
    }


sem_eval = evaluate_retrieval(EVAL_SET, retrieve_semantic, "Semantic", top_k=5)
kw_eval = evaluate_retrieval(EVAL_SET, retrieve_keyword, "Keyword", top_k=5)

print("RETRIEVAL EVALUATION: SEMANTIC vs KEYWORD")
print("=" * 60)
print(f"{'Metric':<20} {'Semantic':>12} {'Keyword':>12}")
print("-" * 60)
print(f"{'Recall@5':<20} {sem_eval['recall_at_k']:>11.1%} {kw_eval['recall_at_k']:>11.1%}")
print(f"{'MRR':<20} {sem_eval['mrr']:>11.3f} {kw_eval['mrr']:>11.3f}")
print(f"{'Questions':<20} {sem_eval['n']:>12} {kw_eval['n']:>12}")

## 26. Detailed Comparison

In [ ]:
print("\nPER-QUERY COMPARISON")
print("=" * 90)

sem_wins = 0
kw_wins = 0
ties = 0

for sd, kd in zip(sem_eval["details"], kw_eval["details"]):
    s_rank = sd["rank"] or 999
    k_rank = kd["rank"] or 999

    if s_rank < k_rank:
        winner = "SEM"
        sem_wins += 1
    elif k_rank < s_rank:
        winner = "KEY"
        kw_wins += 1
    else:
        winner = "TIE"
        ties += 1

    s_status = f"@{sd['rank']}" if sd['hit'] else "MISS"
    k_status = f"@{kd['rank']}" if kd['hit'] else "MISS"
    print(f"  [{winner}] Sem={s_status:>5} Key={k_status:>5} | {sd['query'][:55]}")

print(f"\nSemantic wins: {sem_wins} | Keyword wins: {kw_wins} | Ties: {ties}")

## 27. Evaluation at Different k

In [ ]:
print("RECALL AT DIFFERENT K")
print("=" * 55)
print(f"{'k':>3} | {'Sem Recall':>12} {'Sem MRR':>10} | {'KW Recall':>12} {'KW MRR':>10}")
print("-" * 55)
for k in [1, 3, 5, 10]:
    s = evaluate_retrieval(EVAL_SET, retrieve_semantic, "Sem", top_k=k)
    kw = evaluate_retrieval(EVAL_SET, retrieve_keyword, "KW", top_k=k)
    print(f"  {k:>1} | {s['recall_at_k']:>11.1%} {s['mrr']:>9.3f} | {kw['recall_at_k']:>11.1%} {kw['mrr']:>9.3f}")

## 28. Fix Quality Evaluation (LLM-as-Judge)

In [ ]:
JUDGE_SYSTEM = "You evaluate support ticket fix suggestions for quality."

JUDGE_PROMPT = """Evaluate this support fix suggestion.

NEW TICKET: {ticket}
SUGGESTED FIX: {fix}
TICKETS CITED: {citations}

Score each dimension 1-5:
- actionability: can the agent follow these steps immediately?
- relevance: does the fix address the user's actual problem?
- completeness: does it cover all aspects of the issue?
- citation_accuracy: are the cited tickets actually relevant?
- tone: is the fix professional and empathetic?

Return ONLY JSON:
{{"actionability": N, "relevance": N, "completeness": N,
  "citation_accuracy": N, "tone": N,
  "overall": N, "notes": "brief explanation"}}"""

judge_samples = [
    (t1, r1),
    (t2, r2),
    (t3, r3),
]

print("FIX QUALITY EVALUATION (LLM-as-Judge)")
print("=" * 80)

for ticket, result in judge_samples:
    citations = ", ".join(result.get("tickets_used", [])[:4])
    resp = ask(
        JUDGE_PROMPT.format(
            ticket=ticket[:200],
            fix=result.get("suggested_fix", "")[:400],
            citations=citations,
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  Ticket: {ticket[:60]}...")
        for dim in ["actionability", "relevance", "completeness", "citation_accuracy", "tone"]:
            val = scores.get(dim, "?")
            bar = "*" * int(val) if isinstance(val, (int, float)) else "?"
            print(f"    {dim:20s}: {val}/5 {bar}")
        print(f"    {'overall':20s}: {scores.get('overall', '?')}/5")
        if scores.get("notes"):
            print(f"    Notes: {scores['notes'][:80]}")

---

## 29. Common Mistakes

| Mistake | Why It's Bad | Better Approach |
|---------|-------------|----------------|
| Using only keyword search | Misses paraphrases: "crash" ≠ "freeze" ≠ "not responding" | Semantic search or hybrid |
| Embedding title only | User's description has the key symptoms | Embed title + description together |
| No metadata filtering | Returns irrelevant tickets from other products | Filter by product, severity, category |
| Trusting low-similarity matches | sim < 0.3 means the match is noise | Set a minimum similarity threshold |
| Showing raw past resolutions | Past resolution is for a different user's context | LLM synthesizes an adapted fix |
| No evaluation | Can't tell if retrieval is getting better or worse | Build ground-truth pairs, track Recall@k |

## 30. Production Improvement Ideas

1. **Hybrid search** — combine BM25 keyword search (great for error codes) with semantic search (great for paraphrases)
2. **Auto-triage** — classify severity and route to the right team automatically
3. **Resolution feedback** — let agents mark if the suggestion was helpful; use for fine-tuning
4. **Incremental indexing** — re-embed new tickets as they're resolved so the knowledge base grows
5. **Multi-turn conversation** — ask the user clarifying questions when the ticket is vague
6. **Customer self-serve** — expose this as a search in the help center so users find fixes themselves
7. **Confidence thresholds** — auto-suggest fixes for high-confidence matches, route to human for low

## 31. Exercises

### Exercise 1
Add 5 new tickets for a new product called **MobileApp**. Create evaluation pairs and measure if recall changes.

### Exercise 2
Implement a **hybrid retrieval** function that combines semantic similarity (70% weight) with keyword TF-IDF similarity (30% weight). Compare against pure semantic on the eval set.

### Exercise 3
Build a **confidence gate**: if the best semantic match has similarity < 0.4, return "No confident match found — escalate to a human agent" instead of generating a fix.

### Mini Challenge
Build a ticket de-duplication system: when a new ticket comes in, check if it's a duplicate of an open ticket (similarity > 0.85). If so, merge them and notify the user that their issue is already being worked on.

In [ ]:
print("SESSION SUMMARY")
print("=" * 60)
print(f"Tickets indexed: {len(TICKETS)}")
print(f"Products: {len(set(t['product'] for t in TICKETS))}")
print(f"Evaluation pairs: {len(EVAL_SET)}")
print(f"\nRetrieval results (Recall@5 / MRR):")
print(f"  Semantic: {sem_eval['recall_at_k']:.1%} / {sem_eval['mrr']:.3f}")
print(f"  Keyword:  {kw_eval['recall_at_k']:.1%} / {kw_eval['mrr']:.3f}")
print(f"\nCapabilities built:")
print(f"  - Ticket embedding + ChromaDB indexing")
print(f"  - Semantic retrieval (embedding cosine sim)")
print(f"  - Keyword retrieval (TF-IDF baseline)")
print(f"  - Side-by-side semantic vs keyword comparison")
print(f"  - LLM fix suggestion with ticket citations")
print(f"  - Retrieval evaluation (Recall@k, MRR)")
print(f"  - Fix quality evaluation (LLM-as-judge)")

## 32. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Semantic search finds meaning**, keyword search finds words — use both |
| 2 | Semantic wins for paraphrases: "crash" ≈ "freeze" ≈ "hangs" ≈ "not responding" |
| 3 | Keyword wins for exact codes: `ERR-4021`, product names, version numbers |
| 4 | **Embed title + description** together — title alone loses symptom details |
| 5 | **Metadata filters** (product, severity) dramatically improve precision |
| 6 | **Synthesize** past resolutions instead of showing raw text — context differs |
| 7 | **Cite past tickets** so agents can verify the suggestion |
| 8 | **Low similarity = low confidence** — set thresholds and escalate to humans |
| 9 | **Evaluation is essential** — Recall@k and MRR on ground-truth pairs |
| 10 | **Hybrid search** (semantic + keyword) is the production best practice |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*